# PROJECT PIPELINE: MACHINE LEARNING–BASED ANALYSIS OF SPECIFIC CAPACITANCE ACROSS VARYING PANI CONCENTRATION

## 1. ENVIRONMENT SETUP AND DEPENDENCIES

### 1.1 VIRTUAL ENVIRONMENT PACKAGES (REQUIREMENTS.TXT)

THE FOLLOWING PACKAGES MUST BE INSTALLED IN YOUR VENV TO RUN THIS PIPELINE:

- PANDAS>=1.5.0
- NUMPY>=1.23.0
- MATPLOTLIB>=3.6.0
- SEABORN>=0.12.0
- SCIKIT-LEARN>=1.2.0
- XGBOOST>=1.7.0
- LIGHTGBM>=3.3.0
- SHAP>=0.41.0
- SCIPY>=1.9.0

### 1.2 GLOBAL IMPORTS AND VISUALIZATION STANDARDS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from scipy import stats
import os
from IPython.display import display

warnings.filterwarnings('ignore')

In [2]:
# -----------------------------------------------------------------------------
# GLOBAL VISUALIZATION STANDARDS
# -----------------------------------------------------------------------------
plt.rcParams.update({
    'figure.figsize': (8, 6),
    'figure.dpi': 500,
    'axes.grid': True,
    'font.weight': 'bold',
    'axes.labelweight': 'bold',
    'axes.titleweight': 'bold',
    'font.size': 12
})

## 2. DATA LOADING AND VALIDATION

### 2.1 LOAD DATASETS, CLEAN UNNAMED COLUMNS, AND DISPLAY SHAPE

In [3]:
import pandas as pd
import os

def load_and_interpret_pani_data(cv_dir: str, gcd_dir: str, concentrations: list) -> dict:
    """
    BATCH LOADS CV AND GCD DATASETS ACROSS VARIOUS PANI CONCENTRATIONS.
    CLEANS ARTIFACTS, INJECTS TARGET FEATURES, AND PERFORMS METADATA ANALYSIS.
    
    PARAMETERS:
    - cv_dir: STRING PATH TO CV DIRECTORY
    - gcd_dir: STRING PATH TO GCD DIRECTORY
    - concentrations: LIST OF INTEGERS REPRESENTING PANI %
    
    RETURNS:
    - DICTIONARY MAPPING CONCENTRATION -> {'CV': DataFrame, 'GCD': DataFrame}
    """
    data_vault = {}
    
    print("🔋 INITIATING ELECTROCHEMICAL DATA INGESTION...")
    
    for conc in concentrations:
        cv_path = os.path.join(cv_dir, f'AL203-PANI-{conc}.csv')
        gcd_path = os.path.join(gcd_dir, f'AL203-PANI-{conc}.csv')
        
        try:
            cv_df = pd.read_csv(cv_path)
            gcd_df = pd.read_csv(gcd_path)
            
            # DROP UNNAMED ARTIFACT COLUMNS
            cv_df = cv_df.loc[:, ~cv_df.columns.str.contains('^Unnamed')]
            gcd_df = gcd_df.loc[:, ~gcd_df.columns.str.contains('^Unnamed')]
            
            # INJECT CONCENTRATION AS A NUMERICAL FEATURE FOR PHASE 2 MERGE
            cv_df['PANI_Concentration'] = conc
            gcd_df['PANI_Concentration'] = conc
            
            # STORE IN VAULT
            data_vault[conc] = {'CV': cv_df, 'GCD': gcd_df}
            
            # --- DATA INTERPRETATION & EDA ---
            print(f"\n{'='*55}")
            print(f"🚀 CONCENTRATION LEVEL: {conc}% PANI")
            print(f"{'='*55}")
            
            print("\n[CV METADATA]")
            print(f"SHAPE: {cv_df.shape}")
            cv_df.info()
            
            print("\n[GCD METADATA]")
            print(f"SHAPE: {gcd_df.shape}")
            gcd_df.info()
            
        except FileNotFoundError as e:
            print(f"\n⚠️ FILE NOT FOUND EXPERIMENT ROUTE FAILED FOR {conc}%: {e}")
            
    return data_vault

# ==========================================
# EXECUTE DATA LOADING ARCHITECTURE
# ==========================================
# DEFINE THE CONCENTRATION ARRAY
pani_levels = [0, 5, 10, 15, 20]

# EXECUTE (ENSURE DIRECTORIES END WITHOUT A TRAILING SLASH FOR os.path.join)
data_vault = load_and_interpret_pani_data(
    cv_dir='../../DATASET/DATA/PANI/CV',
    gcd_dir='../../DATASET/DATA/PANI/GCD',
    concentrations=pani_levels
)

🔋 INITIATING ELECTROCHEMICAL DATA INGESTION...

🚀 CONCENTRATION LEVEL: 0% PANI

[CV METADATA]
SHAPE: (4780, 6)
<class 'pandas.DataFrame'>
RangeIndex: 4780 entries, 0 to 4779
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   PANI                4780 non-null   int64  
 1   Scan_Rate           4780 non-null   int64  
 2   Potential           4780 non-null   float64
 3   Current             4780 non-null   float64
 4   CS                  4780 non-null   float64
 5   PANI_Concentration  4780 non-null   int64  
dtypes: float64(3), int64(3)
memory usage: 224.2 KB

[GCD METADATA]
SHAPE: (878, 6)
<class 'pandas.DataFrame'>
RangeIndex: 878 entries, 0 to 877
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   PANI                878 non-null    int64  
 1   Current_Density     878 non-null    float64
 2   Time                878 